In [18]:
from unsloth import FastLanguageModel
import os

checkpoint_dir = "outputs/qwen25-7b-lora-32k-7ep-r64"
os.path.exists(checkpoint_dir)

True

In [19]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(checkpoint_dir),
    max_seq_length=32768,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)

print("Model ready for inference!")

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model ready for inference!


In [8]:
# load validation data
validation_data = 'exports/reversed_messages.validation.jsonl'

import json

with open(validation_data, 'r') as f:
    validation_data = [json.loads(line) for line in f]


In [28]:
test_messages = []

item = validation_data[2]
messages = item['messages']
for message in messages[:3]:
    test_messages.append(message)
    print(f'{message["role"].upper()}: {message["content"][:100]} ...')


ASSISTANT: alright now we have to plan for converting the entire dataset and not just the single_department ... ...
USER: Now let me understand the current dataset structure and conversion context:

Let me understand the s ...
ASSISTANT: 1. B
2. A
3. A
4. the goal details list will be empty! ...


In [29]:
# generate response
prompt = tokenizer.apply_chat_template(
    test_messages[:-1],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# Decode response (only the new tokens)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

generated_message = response.strip()

print(f'Generated message: {generated_message}')

print(f'Actual message: {test_messages[-1]["content"]}')

Generated message: i would prefer option b .. but i dont know how to do it ... can you look into the codebase and show me the code snippets that will will you to implement this
Actual message: 1. B
2. A
3. A
4. the goal details list will be empty!
